In [1]:
import pyspark
from pyspark.sql import SparkSession


In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/13 17:49:51 WARN Utils: Your hostname, codespaces-dff4ee, resolves to a loopback address: 127.0.0.1; using 10.0.3.32 instead (on interface eth0)
26/03/13 17:49:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/13 17:49:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/13 17:49:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/13 17:49:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
spark

In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-13 17:52:28--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.238.152, 18.239.238.212, 18.239.238.119, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.238.152|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M   295MB/s    in 0.2s    

2026-03-13 17:52:29 (295 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [6]:
yellow_path = "yellow_tripdata_2025-11.parquet"

df_yellow = spark.read.parquet(yellow_path)
df_yellow.show(5)

[Stage 1:>                                                          (0 + 1) / 1]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [7]:
df_yellow_repart = df_yellow.repartition(4)
output_path = "data/pq/yellow/2025-11/"

df_yellow_repart.write.mode("overwrite").parquet(output_path)

In [9]:
import os
parquet_files = [f for f in os.listdir(output_path) if f.endswith(".parquet")]
sizes_bytes = [os.path.getsize(os.path.join(output_path, f)) for f in parquet_files]

avg_size_mb = sum(sizes_bytes) / len(sizes_bytes) / (1024*1024)
print(f"Average Parquet file size: {avg_size_mb:.2f} MB")

Average Parquet file size: 25.33 MB


In [10]:
from pyspark.sql.functions import col, to_date

# Filter for trips starting on 2025-11-15
trips_nov15 = df_yellow.filter(
    to_date(col("tpep_pickup_datetime")) == "2025-11-15"
)

# Count the trips
num_trips_nov15 = trips_nov15.count()
print(num_trips_nov15)

162604


In [11]:
from pyspark.sql.functions import col, unix_timestamp, max

# Compute trip duration in hours
df_yellow_with_duration = df_yellow.withColumn(
    "trip_duration_hours",
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 3600
)

# Find the maximum duration
longest_trip = df_yellow_with_duration.select(max("trip_duration_hours")).collect()[0][0]
print(longest_trip)


90.64666666666666


In [12]:
df_zones = spark.read.option("header", True).csv("taxi_zone_lookup.csv")
df_zones.createOrReplaceTempView("zones")

In [13]:
from pyspark.sql.functions import col, count

# Count trips per pickup location
pickup_counts = df_yellow.groupBy("PULocationID").agg(count("*").alias("trip_count"))

# Join with zone names
pickup_with_names = pickup_counts.join(df_zones, pickup_counts.PULocationID == df_zones.LocationID, "left")

# Find the zone with the least trips
least_frequent_zone = pickup_with_names.orderBy(col("trip_count").asc()).select("Zone").first()[0]
print(least_frequent_zone)

[Stage 13:>                                                         (0 + 2) / 2]

Governor's Island/Ellis Island/Liberty Island
